# Dataset Exploration and Visualization

This notebook demonstrates how to:
1. Load and inspect the Aggressive Poses Dataset
2. Visualize labeled images with bounding boxes
3. Analyze class distribution and image statistics
4. Preview pose keypoint extraction

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path('.').parent.resolve()
sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import Counter
import yaml

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Dataset Configuration

Set the path to your dataset:

In [ ]:
# Update this path to your dataset location
DATA_ROOT = Path(r'C:\Users\Nitin N\Aggressive_Poses_Dataset')
# Or for Linux:
# DATA_ROOT = Path('/path/to/Aggressive_Poses_Dataset')

images_dir = DATA_ROOT / 'images'
labels_dir = DATA_ROOT / 'labels'
yaml_path = DATA_ROOT / 'data.yaml'

print(f"Dataset root: {DATA_ROOT}")
print(f"Images dir exists: {images_dir.exists()}")
print(f"Labels dir exists: {labels_dir.exists()}")

## 2. Load and Count Files

In [ ]:
# Get image files
image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
image_files = []
for ext in image_extensions:
    image_files.extend(list(images_dir.glob(f'*{ext}')))
    image_files.extend(list(images_dir.glob(f'*{ext.upper()}')))

# Get label files
label_files = list(labels_dir.glob('*.txt'))

print(f"Total images: {len(image_files)}")
print(f"Total labels: {len(label_files)}")

# Check for matches
image_stems = set(f.stem for f in image_files)
label_stems = set(f.stem for f in label_files)

matched = image_stems & label_stems
images_only = image_stems - label_stems
labels_only = label_stems - image_stems

print(f"\nMatched pairs: {len(matched)}")
print(f"Images without labels: {len(images_only)}")
print(f"Labels without images: {len(labels_only)}")

## 3. Load data.yaml Configuration

In [ ]:
if yaml_path.exists():
    with open(yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    
    print("data.yaml contents:")
    for key, value in data_config.items():
        print(f"  {key}: {value}")
    
    class_names = data_config.get('names', ['aggressive_pose'])
    num_classes = data_config.get('nc', 1)
else:
    print("data.yaml not found")
    class_names = ['aggressive_pose']
    num_classes = 1

print(f"\nClass names: {class_names}")

## 4. Parse YOLO Labels and Analyze

In [ ]:
def parse_yolo_label(label_path):
    """Parse YOLO format label file."""
    annotations = []
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                width = float(parts[3])
                height = float(parts[4])
                annotations.append((class_id, x_center, y_center, width, height))
    return annotations

# Analyze all labels
class_counts = Counter()
boxes_per_image = []
box_widths = []
box_heights = []

for stem in matched:
    label_path = labels_dir / f'{stem}.txt'
    annotations = parse_yolo_label(label_path)
    
    boxes_per_image.append(len(annotations))
    
    for class_id, x_c, y_c, w, h in annotations:
        class_counts[class_id] += 1
        box_widths.append(w)
        box_heights.append(h)

print(f"Total annotations: {sum(class_counts.values())}")
print(f"\nClass distribution:")
for class_id, count in sorted(class_counts.items()):
    name = class_names[class_id] if class_id < len(class_names) else f'class_{class_id}'
    print(f"  {name}: {count}")

print(f"\nBoxes per image:")
print(f"  Mean: {np.mean(boxes_per_image):.2f}")
print(f"  Min: {min(boxes_per_image)}")
print(f"  Max: {max(boxes_per_image)}")

## 5. Visualize Statistics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Class distribution
ax = axes[0, 0]
classes = [class_names[c] if c < len(class_names) else f'class_{c}' for c in sorted(class_counts.keys())]
counts = [class_counts[c] for c in sorted(class_counts.keys())]
ax.bar(classes, counts, color='steelblue')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_title('Class Distribution')

# Boxes per image
ax = axes[0, 1]
ax.hist(boxes_per_image, bins=20, color='coral', edgecolor='black')
ax.set_xlabel('Number of Boxes')
ax.set_ylabel('Frequency')
ax.set_title('Boxes per Image')

# Box sizes
ax = axes[1, 0]
ax.hist(box_widths, bins=30, alpha=0.7, label='Width', color='blue')
ax.hist(box_heights, bins=30, alpha=0.7, label='Height', color='orange')
ax.set_xlabel('Size (normalized)')
ax.set_ylabel('Frequency')
ax.set_title('Box Size Distribution')
ax.legend()

# Width vs Height scatter
ax = axes[1, 1]
ax.scatter(box_widths, box_heights, alpha=0.5, color='purple')
ax.set_xlabel('Box Width')
ax.set_ylabel('Box Height')
ax.set_title('Box Width vs Height')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 6. Visualize Sample Images with Bounding Boxes

In [ ]:
def visualize_image_with_boxes(image_path, label_path, class_names):
    """Visualize image with YOLO bounding boxes."""
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    annotations = parse_yolo_label(label_path)
    
    # Draw boxes
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    
    for class_id, x_c, y_c, bw, bh in annotations:
        # Convert to pixel coordinates
        x1 = int((x_c - bw/2) * w)
        y1 = int((y_c - bh/2) * h)
        x2 = int((x_c + bw/2) * w)
        y2 = int((y_c + bh/2) * h)
        
        color = tuple(int(c * 255) for c in colors[class_id % 10][:3])
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
        
        label = class_names[class_id] if class_id < len(class_names) else f'class_{class_id}'
        cv2.putText(img_rgb, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    return img_rgb

# Display sample images
sample_stems = sorted(list(matched))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, stem in enumerate(sample_stems):
    # Find image file
    img_path = None
    for ext in image_extensions:
        test_path = images_dir / f'{stem}{ext}'
        if test_path.exists():
            img_path = test_path
            break
    
    if img_path:
        label_path = labels_dir / f'{stem}.txt'
        img_vis = visualize_image_with_boxes(img_path, label_path, class_names)
        axes[idx].imshow(img_vis)
        axes[idx].set_title(stem)
        axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 7. Preview Pose Extraction (MediaPipe)

In [ ]:
try:
    import mediapipe as mp
    
    mp_pose = mp.solutions.pose
    mp_draw = mp.solutions.drawing_utils
    
    # Initialize pose model
    pose = mp_pose.Pose(
        static_image_mode=True,
        model_complexity=1,
        min_detection_confidence=0.5
    )
    
    # Process sample images
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, stem in enumerate(sample_stems):
        img_path = None
        for ext in image_extensions:
            test_path = images_dir / f'{stem}{ext}'
            if test_path.exists():
                img_path = test_path
                break
        
        if img_path:
            img = cv2.imread(str(img_path))
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            results = pose.process(img_rgb)
            
            if results.pose_landmarks:
                mp_draw.draw_landmarks(
                    img_rgb,
                    results.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    landmark_drawing_spec=mp_draw.DrawingSpec(color=(255, 0, 0), thickness=2, circle_radius=2),
                    connection_drawing_spec=mp_draw.DrawingSpec(color=(0, 255, 0), thickness=2)
                )
            
            axes[idx].imshow(img_rgb)
            axes[idx].set_title(f'{stem} - Pose Detected: {results.pose_landmarks is not None}')
            axes[idx].axis('off')
    
    pose.close()
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("MediaPipe not installed. Install with: pip install mediapipe")

## 8. Next Steps

After exploring the dataset, run the following pipeline:

```bash
# 1. Split dataset
python split_dataset.py --data_root /path/to/dataset --output_dir yolo_split

# 2. Extract poses
python extract_mediapipe_pose.py --data_root yolo_split --output_dir pose_npy --splits train val test

# 3. Build sequences
python build_sequences.py --pose_dir pose_npy --output_dir sequences --T 16 --stride 8

# 4. Train model
python train_lstm.py --data sequences --epochs 60 --batch_size 32
```